## 1. Introduction

$T_z$ and $T_x$ lifetimes optimization model. The functions $T_z$ and $T_x$ are defined in a similar way to that described in the original notebook of the challenge. 

The main task is to maximize the lifetimes of $T_z$ and $T_x$ at the same time, while keeping the value of bias (defined as $\eta$) at ~$320$ as described in the challenge notebook. Consdering that the Hamiltonian of the system is defined as $H=g_2^*a^2b^{\dagger}+g_2(a^{\dagger})^2b-\epsilon_db-\epsilon_b^{\dagger}$, we can define the function of $T_z$ and $T_x$ as 
$$f(T_x, T_z, \eta)=k_1T_x+k_2T_z-k_3|\eta-320|$$
Where $k-1$, $k-2$, $k-3$ are some constants. 

## 2. End-to-End Measurements ##

As the basis of our optimization function, we are using a modified version of the already provided CMAS-ES algorithm as well as the PPO, and will, in the end, compare their respective results for $g_2$, $\epsilon_d$, $T_x$, and $T_z$ to assess their effectiveness and their accuracy. 

### CMAS-ES based optimizer ###

The CMA-ES optimizer is a population-based search algorithm that learns a probability distribution over good 
parameter values for $g_2$ and $\epsilon_d$. 

Key steps in CMA-ES:

1. Initialize a multivariate Gaussian over the two real parameters $[g_2, \epsilon_d]$.
   - The mean starts at a reasonable operating point (e.g. $[0.2, 4.0]$).
   - The covariance is initialized from the step-size parameter (sigma), which sets the initial search radius.

2. In each generation, we sample a population of candidate parameter sets from this Gaussian.
   - Each candidate is a proposed pair $(g_2, \epsilon_d)$.
   - Candidates are clipped to a valid physical range before evaluation.

3. Evaluate each candidate using the lifetime model.
   - For each $(g_2, \epsilon_d)$, we simulate the cat qubit dynamics and extract $T_z$ and $T_x$.
   - The two lifetimes are combined into a scalar loss function that penalizes deviation from the target bias.
   - Because each simulation is expensive, we cache repeated evaluations and avoid recomputing identical points.

4. Rank the candidates and update the Gaussian distribution.
   - Better-performing candidates pull the mean toward high-value regions.
   - The covariance matrix is adapted so the optimizer can take larger steps in uncertain directions and smaller steps in promising directions.
   - This adaptation is the core of CMA-ES: it learns both where to search and how wide the search should be.

5. Repeat sampling, evaluation, and update for many generations.
   - The distribution gradually concentrates around the best found $(g_2, \epsilon_d)$.
   - The mean of the distribution after optimization is used as the final answer.

In our project, CMA-ES is the direct optimization baseline: it searches the parameter space for the best fixed 
pair of control knobs. We also compare this with PPO, which learns an adaptive policy over knob adjustments. 
In the end, we compare both methods on the same metrics: $g_2$, $\epsilon_d$, $T_x$, and $T_z$.